# Ejemplo de Control Robusto usando SMC

Este ejemplo utiliza control por modo deslizante SMC (Sliding Mode Control) asumiendo un modelo dinámico estimado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Dinámica del Robot

In [ ]:
class Robot2D:
    """
    Robot con 2 grados de libertad
    """
    def __init__(self, dt=0.005, X0=None):
        # Reales
        self.m1 = 1.0
        self.m2 = 0.8
        self.l1 = 1.0
        self.l2 = 0.8
        self.lc1, self.lc2 = 0.5, 0.4
        self.I1, self.I2 = 0.12, 0.08
        self.g = 9.81
        self.X = 0
        self.dt = dt
        if X0 is not None:
            self.X = X0

    def set_state(self, X):
        """
        Establece el estado del sistema (q1, q2, dq1, dq2)
        """
        self.X = X

    def get_state(self):
        """
        Recupera el estado actual (q1, q2, dq1, dq2)
        """
        return self.X.copy()

    def M_(self, q=None):
        """
        Retorna la matriz de inercia M(q)
        """
        if (q is None):
            q1 = self.X[0]; q2 = self.X[1]
        else:
            q1 = q[0]; q2 = q[1]
        a = self.I1 + self.I2 + self.m1*self.lc1**2 + \
            self.m2*(self.l1**2 + self.lc2**2)
        b = self.m2*self.l1*self.lc2
        d = self.I2 + self.m2*self.lc2**2
        c2 = np.cos(q2)
        return np.array([[a + 2*b*c2, d + b*c2],
                         [d + b*c2,   d       ]])

    def C_(self, q=None, dq=None):
        """
        Retorna la matriz de Coriolis y fuerza centrífuga C(q,dq)
        """
        if q is None:
            q1, q2 = self.X[0], self.X[1]
        else:
            q1, q2 = q[0], q[1]
        if dq is None:
            dq1, dq2 = self.X[2], self.X[3]
        else:
            dq1, dq2 = dq[0], dq[1]
        c = self.m2*self.l1*self.lc2
        s2 = np.sin(q2)
        return np.array([[-c*s2*dq2, -c*s2*(dq1 + dq2)],
                         [ c*s2*dq1,  0.0               ]])

    def g_(self, q=None):
        """
        Retorna el vector de gravedad g(q)
        """
        if (q is None):
            q1, q2 = self.X[0], self.X[1]
        else:
            q1, q2 = q[0], q[1]
        g1 = (self.m1*self.lc1 + self.m2*self.l1)*self.g*np.cos(q1) + \
              self.m2*self.lc2*self.g*np.cos(q1 + q2)
        g2 = self.m2*self.lc2*self.g*np.cos(q1 + q2)
        return np.array([g1, g2])

    def simulate(self, u, d):
        """
        Simulación:
        Retorna el estado X=(q1,q2,dq1,dq2) dado un control u y un disturbio d
        Utiliza integración de Euler
        """
        q  = self.X[0:2]
        dq = self.X[2:4]
        M = self.M_(); C = self.C_(); g = self.g_()
        ddq = np.linalg.solve(M, (u + d - C@dq - g))
        dX  = np.hstack((dq,ddq))
        # Integración de Euler
        self.X = self.X + self.dt*dX
        return self.X

    def simulate_rk4(self, u, d):
        """
        Simulación:
        Retorna el estado X=(q1,q2,dq1,dq2) dado un control u y un disturbio d
        Utiliza integración Runge-Kutta 4
        """
        def closed_loop_dynamics_(X, u, d):
            q  = X[0:2]; dq = X[2:4]
            M = self.M_(q); C = self.C_(q, dq); g = self.g_(q)
            ddq = np.linalg.solve(M, (u + d - C@dq - g))
            return np.hstack((dq, ddq))

        f = closed_loop_dynamics_
        k1 = f(self.X, u, d)
        k2 = f(self.X + 0.5*self.dt*k1, u, d)
        k3 = f(self.X + 0.5*self.dt*k2, u, d)
        k4 = f(self.X + self.dt*k3, u, d)
        self.X = self.X + (self.dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        return self.X

### Modelo Estimado del Robot

Devuelve las matrices $M$, $C$ y el vector $g$ estimados. Utiliza parámetros que no son los reales.

In [ ]:
class ModeloRobot:
    """
    Modelo estimado del robot (usando parámetros estimados)
    """
    def __init__(self, m1, m2, l1, l2, lc1, lc2, I1, I2):
        self.m1 = m1
        self.m2 = m2
        self.l1 = l1
        self.l2 = l2
        self.lc1 = lc1
        self.lc2 = lc2
        self.I1 = I1
        self.I2 = I2
        self.g = 9.81

    def get_M(self, q):
        q1, q2 = q
        a = self.I1 + self.I2 + self.m1*self.lc1**2 + self.m2*(self.l1**2 + self.lc2**2)
        b = self.m2*self.l1*self.lc2
        d = self.I2 + self.m2*self.lc2**2
        c2 = np.cos(q2)
        return np.array([[a + 2*b*c2, d + b*c2],
                         [d + b*c2,   d       ]])

    def get_C(self, q, dq):
        q1, q2 = q
        dq1, dq2 = dq
        c = self.m2*self.l1*self.lc2
        s2 = np.sin(q2)
        return np.array([[-c*s2*dq2, -c*s2*(dq1 + dq2)],
                         [ c*s2*dq1,  0.0               ]])

    def get_g(self, q):
        q1, q2 = q
        g1 = (self.m1*self.lc1 + self.m2*self.l1)*self.g*np.cos(q1) + \
             self.m2*self.lc2*self.g*np.cos(q1 + q2)
        g2 = self.m2*self.lc2*self.g*np.cos(q1 + q2)
        return np.array([g1, g2])

### Funciones auxiliares

* Señal de disturbios que se añadirá al modelo dinámico
* Saturación continua

In [ ]:
def disturbios(t, q, dq):
    """
    Perturbación acotada: ondas + fricción de Coulomb
    """
    d_bounded = np.array([0.4*np.sin(0.8*t), 0.3*np.cos(0.6*t)])
    fric_coul = 0.2 * np.sign(dq)
    return d_bounded + fric_coul

def sat(x, phi):
    """
    Saturación continua (anti-chattering): sat(x/phi) = sgn(x) si |x| >> phi
    """
    return np.clip(x / np.maximum(phi, 1e-9), -1.0, 1.0)

### Trayectoria de referencia

In [ ]:
def ref_traj(dt, tf):
    """
    Trayectoria de referencia
    Retorna: q_d(t), dq_d(t), ddq_d(t)
    """
    t = np.arange(0, tf, dt)
    q_d = np.array([ 0.5*np.sin(0.5*t), 0.4*np.cos(0.7*t) ])
    dq_d = np.array([ 0.5*0.5*np.cos(0.5*t), -0.4*0.7*np.sin(0.7*t) ])
    ddq_d = np.array([ -0.5*(0.5**2)*np.sin(0.5*t), -0.4*(0.7**2)*np.cos(0.7*t)])

    return q_d, dq_d, ddq_d, t

In [ ]:
tfinal = 12.0
dt = 0.001

q_des, dq_des, ddq_des, t = ref_traj(dt, tfinal)

# Gráfica de la trayectoria
plt.figure(figsize=(8,3))
plt.plot(t, q_des[0], t, q_des[1])
plt.legend(['$q_0$', '$q_1$'])
plt.title("Trayectoria de referencia")
plt.xlabel("Tiempo [s]"); plt.ylabel("Posición")
plt.grid()

### Simulación

In [ ]:
# Número de iteraciones
N = int(tfinal/dt)
# Variables para almacenar estados y torques
Xt = np.zeros((4,N))   # Almacena estados
Ut = np.zeros((2,N))   # Almacena las señales de control
St = np.zeros((2,N))   # Almacena la superficie de deslizamiento

# Robot simulado
robot = Robot2D(dt)

# Modelo estimado (con incertidumbre: +10% masas, +5% longitudes)
m1h, m2h = 1.1, 0.88
l1h, l2h = 1.05, 0.84
lc1h, lc2h = 0.525, 0.42
I1h, I2h = 0.12*1.10, 0.08*1.10
modelo_estimado = ModeloRobot(m1h, m2h, l1h, l2h, lc1h, lc2h, I1h, I2h)

# Ganancias del controlador
Lambda = np.diag([8.0, 8.0])        # Dinámica en deslizamiento
K_lin  = np.diag([4.0, 4.0])        # Amortiguación lineal sobre s
rho    = np.array([4.0, 4.0])       # Robustez (>= cota de Delta por componente)
phi    = np.array([0.04, 0.04])     # Capa límite para sat (anti-chattering)

In [ ]:
# Condición inicial (alejada de la referencia)
X0 = np.array([0.8, -0.6, 0.0, 0.0])
robot.set_state(X0)
X = robot.get_state()

for k in range(N):

    # Estado actual
    q = X[0:2];  dq = X[2:4]

    # Controlador
    e  = q - q_des[:,k]
    de = dq - dq_des[:,k]
    dq_r  = dq_des[:,k] - Lambda @ e
    ddq_r = ddq_des[:,k] - Lambda @ de
    s = dq - dq_r

    M_est = modelo_estimado.get_M(q)
    C_est = modelo_estimado.get_C(q, dq)
    g_est = modelo_estimado.get_g(q)

    tau_eq = M_est @ ddq_r + C_est @ dq_r + g_est
    # Señal de control
    u = tau_eq - (K_lin@s) - rho*sat(s, phi)

    # Disturbios
    d  = disturbios(k*dt, q, dq)

    # Simulación del robot
    robot.simulate_rk4(u, d)
    X = robot.get_state()

    Xt[:,k] = X
    Ut[:,k] = u
    St[:,k] = s

print("Simulación terminada")

### Figuras resultantes

In [ ]:
plt.figure(figsize=(10,6))
plt.subplot(3,1,1)
plt.plot(t, Xt[0,:], label='q1')
plt.plot(t, q_des[0,:], '--', label='q1_d')
plt.plot(t, Xt[1,:], label='q2')
plt.plot(t, q_des[1,:], '--', label='q2_d')
plt.ylabel('Posición [rad]')
plt.legend(); plt.grid(True)

plt.subplot(3,1,2)
plt.plot(t, St[0,:], label='s1')
plt.plot(t, St[1,:], label='s2')
plt.ylabel('Sliding s')
plt.legend(); plt.grid(True)

plt.subplot(3,1,3)
plt.plot(t, Ut[0,:], label='tau1')
plt.plot(t, Ut[1,:], label='tau2')
plt.ylabel('Torque [Nm]')
plt.xlabel('Tiempo [s]')
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()